**SAPIANS** · SCC5819 · MÓDULO 00 — A BASE SRAG · O ESTUDO GLOBAL

# Modelo do curso com todo o treino elegível

A origem dos dados, a divisão temporal e a escolha do preditor vêm antes dos
métodos de interpretação. Este caderno distingue o estudo histórico amostrado
da nova seleção por log loss e lê o registro verificável da execução completa.

[README do módulo](../README.md) · [protocolo](../COURSE_STUDY.md) ·
[internals](srag_course_study_internals.ipynb)

---

**Objetivos.** Identificar a população e o alvo; separar partições de amostragem;
comparar os candidatos com desbalanceamento tratado explicitamente; reconhecer
o preditor congelado e os limites da avaliação temporal já observada.

In [1]:
%pip install -q numpy pandas pyarrow scikit-learn xgboost imbalanced-learn joblib matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import pathlib
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
RANDOM_STATE = 42
REPO = pathlib.Path(os.environ.get("INTERPRETABLE_ML_REPO", pathlib.Path.cwd())).resolve()
while REPO != REPO.parent and not (REPO / "tools/sapians.py").exists():
    REPO = REPO.parent
if not (REPO / "tools/sapians.py").exists():
    raise RuntimeError("Abra este caderno em um checkout do repositório.")
sys.path.insert(0, str(REPO / "tools"))
import sapians as SP
SP.aplicar()
import srag_51_course_study as C
import srag_52_course_balance as B
RECORD = REPO / "modules/00-dataset/gold/course_study_v2.json"
r = json.loads(RECORD.read_text())
protocol = r["protocol"]
print("Versão:", protocol["version"])
print("Critério principal: menor log loss na validação temporal.")
print("Estado da execução:", r["status"])


Versão: course-full-logloss-v2
Critério principal: menor log loss na validação temporal.
Estado da execução: running


## §1 — Coorte, partições e treino efetivo

O treino completo é uma partição da coorte, não a totalidade de todas as
notificações de SRAG. A referência sem balanceamento usa todas as linhas
admitidas nesse período. Os amostradores transformam somente o treino.


In [3]:
rows = [{"partição": split, **counts} for split, counts in protocol["splits"].items()]
print(SP.tabela(pd.DataFrame(rows)))
print("Coorte Gold:", SP.pt_int(sum(x["n"] for x in protocol["splits"].values())))
print("Entradas do modelo:", SP.pt_int(len(protocol["features"])))
print("Treino histórico amostrado:", SP.pt_int(200_000))
print("Treino elegível da nova referência:", SP.pt_int(protocol["splits"]["train"]["n"]))
print("Alvo: EVOLUCAO código 2 (óbito) versus código 1 (cura); código 3 excluído.")
print("Períodos: treino 2020–2022; validação 2023; avaliação histórica 2024.")

partição         n       events
    test    16.142           NA
   train 1.242.680 390.769,0000
     val    24.148   4.572,0000
Coorte Gold: 1.282.970
Entradas do modelo: 40
Treino histórico amostrado: 200.000
Treino elegível da nova referência: 1.242.680
Alvo: EVOLUCAO código 2 (óbito) versus código 1 (cura); código 3 excluído.
Períodos: treino 2020–2022; validação 2023; avaliação histórica 2024.


**O que olhar.** A redução histórica do treino não fazia parte da PFI. Não
há uma proporção fixa entre as partições; o calendário define os conjuntos.
O código de óbito por outras causas não é convertido em sobrevida.

## §2 — Candidatos e tratamento do desbalanceamento

A referência preserva as frequências do período de treino. Pesos, redução e
SMOTENC restrito são alternativas medidas, não tratamentos obrigatórios para
tornar todas as classes iguais. A avaliação conserva a distribuição real.


In [4]:
specs = pd.DataFrame(protocol["candidates"])
print(SP.tabela(specs.groupby(["strategy", "family"], observed=True).size().rename("configurações").reset_index()))
print("Configurações registradas:", SP.pt_int(len(specs)))
print("Recalibração:", protocol["probability_calibration"])
print("Regra para categorias novas:", protocol["xgb_encoding"])

          strategy   family configurações
             plain   arvore            24
             plain    dummy             1
             plain floresta            10
             plain    logit             7
             plain      lpm             1
             plain      xgb            30
               rus    logit             7
               rus      xgb            30
smotenc_restricted    logit             7
smotenc_restricted      xgb            30
           weights    logit             7
           weights      xgb            30
Configurações registradas: 184
Recalibração: No post-fit recalibration; reliability bins and calibration diagnostics never modify predictions.
Regra para categorias novas: Train-fitted ordinal codes declared categorical to XGBoost; unknown evaluation categories become missing. Numeric missing values remain native.


**O que olhar.** Logística e XGBoost têm o mesmo orçamento de configurações
por estratégia. As outras famílias compõem referências sem balanceamento.
A busca é finita; o vencedor é o melhor entre os candidatos executados.

## §3 — Resultados da validação e escolha

A comparação usa perdas sem arredondamento. As demais métricas e as curvas de
calibração complementam o critério, sem substituí-lo depois dos resultados.


In [5]:
trials = r.get("validation_candidates", [])
if trials:
    table = pd.DataFrame([{"família": x["family"], "estratégia": x["strategy"], "candidato": x["candidate_id"], **{k: x["validation"][k] for k in ("logloss", "auc", "brier", "ap", "mean_predicted", "observed")}} for x in trials])
    table = table.sort_values(["logloss", "candidato"])
    print(SP.tabela(table.groupby(["família", "estratégia"], sort=False).head(1), {"logloss": 6, "auc": 4, "brier": 6, "ap": 4, "mean_predicted": 4, "observed": 4}))
print("Ajustes completos:", SP.pt_int(len(trials)))
selection = r.get("selection")
if selection:
    winner = selection["winner"]
    print("Vencedor:", winner["family"], "/", winner["strategy"])
    print("Log loss de validação:", SP.pt(winner["validation"]["logloss"], 6))
    print("Identificação do modelo:", selection["canonical_model"]["sha256"])
else:
    print("Seleção ainda não congelada: o protocolo exige todos os candidatos completos.")

 família estratégia        candidato  logloss    auc    brier     ap mean_predicted observed
  arvore      plain 9e96a0e9b6b198d4 0,435505 0,7227 0,140626 0,3331         0,2185   0,1893
   logit      plain 1b857d25a5f43661 0,461945 0,7217 0,153292 0,3359         0,2690   0,1893
floresta      plain 39a46c3acdc4fb5b 0,466391 0,7353 0,148613 0,3528         0,2911   0,1893
     lpm      plain 68d02d0c9c6fee85 0,510797 0,7241 0,153743 0,3391         0,2764   0,1893
   dummy      plain 0a8f093e98574dd4 0,525103 0,5000 0,169142 0,1893         0,3145   0,1893
Ajustes completos: 34
Seleção ainda não congelada: o protocolo exige todos os candidatos completos.


In [6]:
if selection:
    best = table.groupby(["família", "estratégia"], sort=False).head(1).sort_values("logloss")
    families = {"dummy": "Baseline", "lpm": "Linear", "logit": "Logística", "arvore": "Árvore", "floresta": "Floresta", "xgb": "XGBoost"}
    strategies = {"plain": "sem balanceamento", "weights": "pesos", "rus": "redução", "smotenc_restricted": "SMOTENC restrito"}
    labels = [families[x] + " · " + strategies[y] for x, y in zip(best["família"], best["estratégia"])]
    fig, ax = plt.subplots(figsize=SP.SLOT)
    colors = [SP.ESCURO if cid == selection["winner"]["candidate_id"] else SP.CINZA for cid in best["candidato"]]
    ax.scatter(best["logloss"], np.arange(len(best)), c=colors, s=42)
    ax.set_yticks(np.arange(len(best)), labels)
    ax.invert_yaxis()
    ax.set_xlabel("Log loss na validação · menor é melhor")
    ax.set_ylabel("")
    for i, loss in enumerate(best["logloss"]):
        ax.annotate(SP.pt(loss, 3), (loss, i), xytext=(7, 0), textcoords="offset points", va="center", fontsize=9)
    ax.margins(x=0.18, y=0.06)
    SP.titulo(fig, "A escolha compara a perda na distribuição real", "§3 · MELHOR CONFIGURAÇÃO DE CADA FAMÍLIA E ESTRATÉGIA")
    SP.salvar(fig, "course-study-v2-validation")
    plt.show()
else:
    print("Gráfico final aguardando todos os candidatos; sem vencedor antecipado.")

Gráfico final aguardando todos os candidatos; sem vencedor antecipado.


## §4 — Avaliação temporal e modelo único

A seleção é congelada antes da previsão no teste. Esse período já foi visto
anteriormente; o resultado é um backtest exploratório, não uma confirmação
independente. As aulas só podem atribuir explicações ao modelo identificado.


In [7]:
evaluation = r.get("evaluation")
if evaluation:
    print(SP.tabela(pd.DataFrame([evaluation["test"]]).drop(columns=["reliability"]), {"logloss": 6, "auc": 4, "brier": 6, "ap": 4}))
    print("Estado do teste:", evaluation["status"])
    print("Seleção mudou depois do teste:", evaluation["selection_changed"])
else:
    print("Teste não avaliado nesta execução: aguardando seleção completa.")
print("Adotar um modelo diferente exige recalcular as explicações dependentes.")

Teste não avaliado nesta execução: aguardando seleção completa.
Adotar um modelo diferente exige recalcular as explicações dependentes.


In [8]:
if evaluation:
    winner_rows = selection["winner"]["validation"]["reliability"]
    test_rows = evaluation["test"]["reliability"]
    fig, (ax, counts_ax) = plt.subplots(1, 2, figsize=SP.SLOT, gridspec_kw={"width_ratios": [1.4, 1]})
    ax.plot([0, 1], [0, 1], color=SP.CINZA, linestyle="--", label="Correspondência ideal")
    for label, bins, color in (("Validação", winner_rows, SP.ESCURO), ("Teste histórico", test_rows, SP.AMBAR)):
        ax.plot([b["mean_predicted"] for b in bins], [b["observed"] for b in bins], marker="o", markersize=4, color=color, label=label)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Probabilidade média prevista", ylabel="Frequência observada de óbito")
    ax.legend(fontsize=9)
    counts_ax.bar([b["bin"] / 10 + 0.05 for b in test_rows], [b["n"] for b in test_rows], width=0.085, color=SP.CINZA)
    counts_ax.set(xlim=(0, 1), xlabel="Faixa de probabilidade", ylabel="Registros no teste")
    SP.titulo(fig, "A calibração confronta risco previsto e frequência observada", "§4 · MESMO PREDITOR CONGELADO · FAIXAS DE LARGURA IGUAL")
    SP.salvar(fig, "course-study-v2-calibration")
    plt.show()
else:
    print("Calibração final aguardando o modelo congelado e sua avaliação temporal.")

Calibração final aguardando o modelo congelado e sua avaliação temporal.


## §5 — O que o livro diz e o que pertence ao estudo do modelo

Molnar apresenta a PFI como avaliação de um preditor já ajustado. A decisão
sobre coorte, treinamento, balanceamento e escolha desse preditor pertence ao
estudo global, não à permutação. Aqui log loss também será a perda da PFI após
a adoção do modelo. Um ranking de importância não escolhe o modelo nem valida
seu uso clínico.

O percurso dos métodos implementados chega à PFI no capítulo 23 de Molnar;
isso não transforma os capítulos intermediários planejados em experimentos
executados. O [relatório ampliado](../../../reports/02-modelo-global-pfi/README.md)
separa resultados históricos e a nova execução.

Fontes: [Molnar, PFI](https://christophm.github.io/interpretable-ml-book/feature-importance.html),
[scikit-learn, escores e objetivo](https://scikit-learn.org/stable/modules/model_evaluation.html#which-scoring-function-should-i-use),
[imbalanced-learn, separação e reamostragem](https://imbalanced-learn.org/stable/common_pitfalls.html).